# 1980s Classic Hits

Análisis inicial del dataset y preparación de variables para responder las consignas de la actividad.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

hits_df = pd.read_csv('1980sClassics.csv')

print(f'Dimensiones del dataset: {hits_df.shape}')
print('\nColumnas:')
print(hits_df.columns.tolist())
print('\nTipos de datos:')
print(hits_df.dtypes)
hits_df.head()

Dimensiones del dataset: (998, 17)

Columnas:
['Track', 'Artist', 'Duration', 'Time_Signature', 'Danceability', 'Energy', 'Key', 'Loudness', 'Mode', 'Speechiness', 'Acousticness', 'Instrumentalness', 'Liveness', 'Valence', 'Tempo', 'Popularity', 'Year']

Tipos de datos:
Track                   str
Artist                  str
Duration                str
Time_Signature        int64
Danceability        float64
Energy              float64
Key                   int64
Loudness            float64
Mode                  int64
Speechiness         float64
Acousticness        float64
Instrumentalness    float64
Liveness            float64
Valence             float64
Tempo               float64
Popularity            int64
Year                  int64
dtype: object


,Track,Artist,Duration,Time_Signature,Danceability,Energy,Key,Loudness,Mode,Speechiness,Acousticness,Instrumentalness,Liveness,Valence,Tempo,Popularity,Year
0,Babe,Styx,3:38,4,0.700,0.582,11,-5.960,0,0.0356,0.05020,0.000000,0.0881,0.785,116.712,96,1980
1,The Rose,Bette Midler,4:04,4,0.264,0.640,8,-6.221,1,0.0442,0.03930,0.000002,0.1510,0.190,84.828,92,1980
2,Cars,Gary Numan,4:08,4,0.338,0.562,9,-7.181,1,0.0290,0.03900,0.000000,0.1070,0.259,149.907,82,1980
3,Magic,Olivia Newton-John,2:17,4,0.911,0.689,1,-6.176,1,0.2650,0.00119,0.000000,0.0704,0.546,140.034,80,1980
4,We Don’t Talk Anymore,Cliff Richard,3:37,4,0.728,0.563,1,-8.053,0,0.1340,0.62100,0.000000,0.1790,0.352,100.017,80,1980


In [2]:
# Transformación de Duration: de mm:ss a segundos
hits_df['Duration_seconds'] = (
    hits_df['Duration'].str.split(':').str[0].astype(int) * 60
    + hits_df['Duration'].str.split(':').str[1].astype(int)
)

print('Ejemplo de transformación de Duration:')
display(hits_df[['Duration', 'Duration_seconds']].head())

Ejemplo de transformación de Duration:


,Duration,Duration_seconds
0,3:38,218
1,4:04,244
2,4:08,248
3,2:17,137
4,3:37,217


In [3]:
# Track y Artist tienen muchas categorías, por lo que one-hot encoding puede generar demasiadas columnas.
# Frequency encoding o target encoding son alternativas más adecuadas.
for columna in ['Track', 'Artist']:
    frecuencia = hits_df[columna].value_counts(normalize=True)
    hits_df[f'{columna}_frequency'] = hits_df[columna].map(frecuencia)

print('Cantidad de categorías:')
print(hits_df[['Track', 'Artist']].nunique())

Cantidad de categorías:
Track     972
Artist    475
dtype: int64


In [4]:
# Comparación de discretización por cuantiles e intervalos de igual amplitud
variables_discretizacion = ['Speechiness', 'Energy', 'Danceability', 'Popularity']
comparacion = []

for columna in variables_discretizacion:
    por_cuantiles = pd.qcut(hits_df[columna], q=4, duplicates='drop').value_counts().sort_index()
    por_amplitud = pd.cut(hits_df[columna], bins=4).value_counts().sort_index()
    comparacion.append({
        'variable': columna,
        'diferencia_maxima': int(abs(por_cuantiles.values - por_amplitud.values).max()),
        'asimetria': hits_df[columna].skew()
    })

comparacion_df = pd.DataFrame(comparacion).sort_values('diferencia_maxima', ascending=False)
display(comparacion_df)
print('La mayor diferencia corresponde a:', comparacion_df.iloc[0]['variable'])

,variable,diferencia_maxima,asimetria
0,Speechiness,694,3.943758
3,Popularity,322,-0.977191
2,Danceability,237,-0.312281
1,Energy,206,-0.440285


La mayor diferencia corresponde a: Speechiness


In [5]:
# Balance de Mode
balance_mode = hits_df['Mode'].value_counts(normalize=True).mul(100).round(2)
print('Balance de Mode (%):')
print(balance_mode)

# Para modelar una variable objetivo relacionada con Mode, conviene usar stratify en el split.
# Si el desempeño de la clase minoritaria fuera bajo, se podrían evaluar SMOTE o undersampling.

Balance de Mode (%):
Mode
1    68.94
0    31.06
Name: proportion, dtype: float64
